# 01 - Data Cleaning
TMDB Reel Insights - load, clean, and build the SQLite database.

## 1. Setup & Load Raw Data

In [1]:
import pandas as pd
import numpy as np
import sqlite3

pd.set_option('display.max_columns', None)

movies = pd.read_csv('../data/movies.csv')
genres = pd.read_csv('../data/genres.csv')
movie_genres = pd.read_csv('../data/movie_genres.csv')
cast = pd.read_csv('../data/cast.csv')
crew = pd.read_csv('../data/crew.csv')
movie_keywords = pd.read_csv('../data/movie_keywords.csv')

tables = {
    'movies': movies, 'genres': genres, 'movie_genres': movie_genres,
    'cast': cast, 'crew': crew, 'movie_keywords': movie_keywords
}
for name, df in tables.items():
    print(f"{name}: {df.shape}")

movies: (2509, 12)
genres: (19, 2)
movie_genres: (6888, 2)
cast: (24917, 6)
crew: (7297, 6)
movie_keywords: (41701, 3)


## 2. Inspect Shapes, Dtypes, Nulls

In [2]:
for name, df in tables.items():
    print(f"\n=== {name} ===")
    print(df.dtypes)
    print("Nulls:\n", df.isna().sum()[df.isna().sum() > 0])


=== movies ===
movie_id               int64
title                    str
original_language        str
release_date             str
runtime                int64
budget               float64
revenue              float64
popularity           float64
vote_average         float64
vote_count             int64
status                   str
overview                 str
dtype: object
Nulls:
 Series([], dtype: int64)

=== genres ===
genre_id      int64
genre_name      str
dtype: object
Nulls:
 Series([], dtype: int64)

=== movie_genres ===
movie_id    int64
genre_id    int64
dtype: object
Nulls:
 Series([], dtype: int64)

=== cast ===
cast_row_id       int64
movie_id          int64
person_id         int64
actor_name          str
character_name      str
cast_order        int64
dtype: object
Nulls:
 character_name    11
dtype: int64

=== crew ===
crew_row_id    int64
movie_id       int64
person_id      int64
person_name      str
job              str
department       str
dtype: object
Nulls:
 Serie

No nulls anywhere. `release_date` needs converting to an actual date. budget/revenue have a lot of zeros - TMDB uses 0 for "unknown", not free, so I'll flag those instead of treating them as real values.

## 3. Fix Data Types

In [3]:
movies['release_date'] = pd.to_datetime(movies['release_date'], errors='coerce')
movies['movie_id'] = movies['movie_id'].astype(int)
movies['runtime'] = movies['runtime'].astype(int)
movies['vote_count'] = movies['vote_count'].astype(int)

# Flag placeholder zero values instead of silently treating them as real numbers
movies['budget_known'] = movies['budget'] > 0
movies['revenue_known'] = movies['revenue'] > 0

print(f"Movies with unknown budget: {(~movies['budget_known']).sum()} "
      f"({(~movies['budget_known']).mean():.1%})")
print(f"Movies with unknown revenue: {(~movies['revenue_known']).sum()} "
      f"({(~movies['revenue_known']).mean():.1%})")
print(f"Rows with unparseable release_date: {movies['release_date'].isna().sum()}")

for df, cols in [(genres, ['genre_id']), (movie_genres, ['movie_id','genre_id']),
                  (cast, ['cast_row_id','movie_id','person_id','cast_order']),
                  (crew, ['crew_row_id','movie_id','person_id']),
                  (movie_keywords, ['movie_id','keyword_id'])]:
    for c in cols:
        df[c] = df[c].astype(int)

movies.head()

Movies with unknown budget: 90 (3.6%)
Movies with unknown revenue: 102 (4.1%)
Rows with unparseable release_date: 0


,movie_id,title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,status,overview,budget_known,revenue_known
0,157336,Interstellar,en,2014-11-05,169,165000000.0,7.466067e+08,73.0452,8.484,40566,Released,The adventures of a group of explorers who mak...,True,True
1,27205,Inception,en,2010-07-15,148,160000000.0,8.390306e+08,55.8497,8.400,39727,Released,"Cobb, a skilled thief who commits corporate es...",True,True
2,24428,The Avengers,en,2012-04-25,143,220000000.0,1.518816e+09,72.2515,8.062,38993,Released,When an unexpected enemy emerges and threatens...,True,True
3,155,The Dark Knight,en,2008-07-16,152,185000000.0,1.004558e+09,60.5510,8.534,36255,Released,Batman raises the stakes in his war on crime. ...,True,True
4,19995,Avatar,en,2009-12-16,162,237000000.0,2.923706e+09,48.1996,7.610,34348,Released,"In the 22nd century, a paraplegic Marine is di...",True,True


Added `budget_known` / `revenue_known` flags. Any profit or ROI calc later should filter on these instead of using budget/revenue as-is.

## 4. Remove Exact Duplicate Rows

In [4]:
for name, df in tables.items():
    before = len(df)
    tables[name] = df.drop_duplicates()
    after = len(tables[name])
    if before != after:
        print(f"{name}: dropped {before - after} exact duplicate rows ({before} -> {after})")

movies, genres, movie_genres, cast, crew, movie_keywords = (
    tables['movies'], tables['genres'], tables['movie_genres'],
    tables['cast'], tables['crew'], tables['movie_keywords']
)

movies: dropped 6 exact duplicate rows (2509 -> 2503)
movie_genres: dropped 10 exact duplicate rows (6888 -> 6878)
cast: dropped 15 exact duplicate rows (24917 -> 24902)
crew: dropped 8 exact duplicate rows (7297 -> 7289)
movie_keywords: dropped 20 exact duplicate rows (41701 -> 41681)


## 5. Remove Orphaned Rows
Drop rows in the child tables that point to a `movie_id` we removed above.

In [5]:
valid_movie_ids = set(movies['movie_id'])
valid_genre_ids = set(genres['genre_id'])

def drop_orphans(df, id_col, valid_ids, name):
    before = len(df)
    out = df[df[id_col].isin(valid_ids)].copy()
    dropped = before - len(out)
    if dropped:
        print(f"{name}: dropped {dropped} orphaned rows (bad {id_col})")
    return out

movie_genres = drop_orphans(movie_genres, 'movie_id', valid_movie_ids, 'movie_genres')
movie_genres = drop_orphans(movie_genres, 'genre_id', valid_genre_ids, 'movie_genres')
cast = drop_orphans(cast, 'movie_id', valid_movie_ids, 'cast')
crew = drop_orphans(crew, 'movie_id', valid_movie_ids, 'crew')
movie_keywords = drop_orphans(movie_keywords, 'movie_id', valid_movie_ids, 'movie_keywords')

print("\nFinal shapes:")
for name, df in [('movies', movies), ('genres', genres), ('movie_genres', movie_genres),
                  ('cast', cast), ('crew', crew), ('movie_keywords', movie_keywords)]:
    print(f"  {name}: {df.shape}")


Final shapes:
  movies: (2503, 14)
  genres: (19, 2)
  movie_genres: (6878, 2)
  cast: (24902, 6)
  crew: (7289, 6)
  movie_keywords: (41681, 3)


## 6. Load Cleaned Tables into SQLite

In [6]:
conn = sqlite3.connect('../data/movies.db')

movies.to_sql('movies', conn, if_exists='replace', index=False)
genres.to_sql('genres', conn, if_exists='replace', index=False)
movie_genres.to_sql('movie_genres', conn, if_exists='replace', index=False)
cast.to_sql('cast', conn, if_exists='replace', index=False)
crew.to_sql('crew', conn, if_exists='replace', index=False)
movie_keywords.to_sql('movie_keywords', conn, if_exists='replace', index=False)

conn.execute("CREATE INDEX IF NOT EXISTS idx_mg_movie ON movie_genres(movie_id)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_mg_genre ON movie_genres(genre_id)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_cast_movie ON cast(movie_id)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_crew_movie ON crew(movie_id)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_kw_movie ON movie_keywords(movie_id)")
conn.commit()

print("Loaded all 6 tables into data/movies.db")

Loaded all 6 tables into data/movies.db


## 7. Test Query — Confirm the Database Loaded Correctly

In [7]:
test = pd.read_sql('''
    SELECT m.title, m.release_date, g.genre_name
    FROM movies m
    JOIN movie_genres mg ON m.movie_id = mg.movie_id
    JOIN genres g ON mg.genre_id = g.genre_id
    ORDER BY m.release_date DESC
    LIMIT 5
''', conn)
test

,title,release_date,genre_name
0,Disclosure Day,2026-06-10 00:00:00,Science Fiction
1,Disclosure Day,2026-06-10 00:00:00,Thriller
2,Backrooms,2026-05-27 00:00:00,Horror
3,Backrooms,2026-05-27 00:00:00,Mystery
4,Backrooms,2026-05-27 00:00:00,Science Fiction


In [8]:
conn.close()
print("Connection closed. data/movies.db is ready for 02_sql_analysis.ipynb")

Connection closed. data/movies.db is ready for 02_sql_analysis.ipynb


## Summary
- Fixed dtypes, removed duplicate rows, removed orphan rows.
- budget/revenue zeros kept but flagged instead of guessed at.
- Data loaded into `movies.db`, ready for the SQL notebook.